In [1]:
# Package import 
import pandas as pd
import numpy as np
import random
from geopy.geocoders import Nominatim
from country_converter import country_converter as coco
from carbon_bombs.utils.logger import LOGGER

from carbon_bombs.io.khune_paper import load_carbon_bomb_gasoil_database
from carbon_bombs.io.rystad import load_rystad_emission_database
from carbon_bombs.io.khune_paper import load_carbon_bomb_gasoil_database
from carbon_bombs.utils.location import get_world_region
from carbon_bombs.utils.location import clean_project_names_with_iso
from carbon_bombs.conf import DATA_SOURCE_PATH
from carbon_bombs.conf import SHEETNAME_RYSTAD_EXPANSION_EMISSION
from carbon_bombs.conf import SHEETNAME_RYSTAD_CB_EMISSION
from carbon_bombs.conf import SHEETNAME_RYSTAD_CB_COMPANY
from carbon_bombs.conf import SHEETNAME_RYSTAD_CB_EMISSION_INFERIOR_1GT


In [2]:
# Add custom functions
def _add_country_lat_long(
    df: pd.DataFrame,
    country_col: str = "country",
    noise_strength: float = 0.05
) -> pd.DataFrame:
    """
    Add Latitude and Longitude columns based on the provided country column.
    Adds slight random noise to coordinates to prevent marker overlap on maps.
    
    Args:
        df: Input DataFrame.
        country_col: Name of the column containing the country.
        noise_strength: Maximum degrees of random noise to add (default ±0.05°).
    """
    # Load reference coordinates once
    country_lat_long_df = pd.read_csv(f"{DATA_SOURCE_PATH}/longitude-latitude.csv")

    # Prepare dictionary for fast lookup
    iso3_to_coords = {
        row["ISO-ALPHA-3"]: (row["Latitude"], row["Longitude"])
        for _, row in country_lat_long_df.iterrows()
    }

    # Convert all country names to ISO3 once
    iso3_series = coco.convert(names=df[country_col], to="ISO3", not_found=None)

    # Define function to get coordinates
    def get_coords(iso3):
        if iso3 in iso3_to_coords:
            lat, lon = iso3_to_coords[iso3]
            lat += random.uniform(-noise_strength, noise_strength)
            lon += random.uniform(-noise_strength, noise_strength)
            return pd.Series([lat, lon])
        else:
            return pd.Series([0.0, 0.0])

    # Apply and expand result into two columns
    df[["latitude", "longitude"]] = pd.Series(iso3_series).apply(get_coords)

    return df

# Load Files relative to gasoil

1 - Emissions file for carbon bombs and expansion projects (Carbon_Bombs_Projects.xlsx)
Fichier initial Carbon_Bombs_Projects.xlsx contenant 3 onglets https://data-for-good.slack.com/archives/C08C639D8HM/p1745590952346059 : 
- Carbon_bombs_1GT : Tout les carbon bombs (projets > 1GtCO2) en version 2 (Avril 2025)
- Carbon_Bombs_Projects : Tout les projets d'expansion > 5mtCO2 entre 2025 et 2050 
- <1GT (no data publication) : Projets dont les émissions sont supérieurs à 0.8GTCO2 pour investiguer les CB de la v1 manquantes (onglet finalement déprécié)
Fichier (Confidential - Carbon Bombs 0.1GT.xlsx) pour investiguer les CB de la v1 aujourd'hui > 0,1 GTCO2 : https://data-for-good.slack.com/archives/C08C639D8HM/p1747312374821879?thread_ts=1747213566.449539&cid=C08C639D8HM
1 onglet : V1_method_0.1GT que j'ai remplacé manuellement dans le fichier Carbon_Bombs_Projects.xlsx avec l'onglet - <1GT (no data publication) pour permettre une comparaison complète  
Fichier avec des données complémentaires sur 38 projets que j'ai inséré manuellement dans le fichier de base (22 non matchés initialement + 16 où les emissions n'étaient pas les bonnes) : https://reclaimfinance.sharepoint.com/:x:/s/ReclaimCloud/EQ_AjVb_ChdPrYOLDaqpS04BW7jSjxL7dgQTwHmyVKOFjw?rtime=3lvVZnHp3Ug (voir Slack : https://data-for-good.slack.com/archives/C08C639D8HM/p1756134477792339?thread_ts=1754847139.766209&cid=C08C639D8HM)
- Ajout manuel de 13 projets de la v1 dont les emissions était manquantes dans l'onglet V1_method_0.1GT via le fichier https://reclaimfinance-my.sharepoint.com/:x:/g/personal/louis-maxence_reclaimfinance_onmicrosoft_com1/EW-P-Zr5n59LqpG0OD-051ABMeIPjFRuaNZc4QWCrLQkgA?rtime=QYgoM3fz3Ug (voir message https://data-for-good.slack.com/archives/C08C639D8HM/p1757519575321379?thread_ts=1756923084.900939&cid=C08C639D8HM) Liste des projets concernés = Khafi, NS Bab (Gasco), AE Yucatan Platform Offshore, MX Gulf Coast Centre Offshore, US West Florida Offshore, Us Baltimore Canyon Offshore, US Liard Shale, CA Kronprins Christian Offshore, GL Lublin Basin Silurian Shale, PL Shtokman, RU Rusanovskoye (Kara Sea), RU Leningradskoye (Kara Sea), RU Taymyr Basin CBM, RU


2 - Emissions between 2020 and 2025 for gasoil carbon bombs 20250718_Confidential - CarbonBombs_Yearly emissions.xlsx https://data-for-good.slack.com/archives/C08CJPQJD19/p1752851371154879





3 - Company related to carbon bombs and expansion projects (20250717_Confidential - Carbon_Bombs_Companies_For D4G.xlsx)  
https://data-for-good.slack.com/archives/C08C639D8HM/p1752768741703839

4 - Banks financing companies (BOCC file) (BOCC25_All FF_Aggregated_D4G.xlsx) Version 2 to be coming with Gogel ID and LEI ID  
https://data-for-good.slack.com/archives/C08C639D8HM/p1750749724360129  
  

Version 2 with GogelId ici : https://data-for-good.slack.com/archives/C08C639D8HM/p1754489711016029?thread_ts=1754236905.591929&cid=C08C639D8HM
Lien vers un share point exporté dans data_sources sous le nom Confidential_BOCC25_All FF_Aggregated_D4G with GOGEL ID.xlsx






New version for gogel_id here avec uniquement des parents_company. 
Message le 16/09/2025 de Maxence. 
https://data-for-good.slack.com/archives/C08C639D8HM/p1758032400664619. 

2 fichiers actualisés :   
20250916_Confidential_BOCC25_All FF_Aggregated_D4G with GOGEL ID : https://reclaimfinance.sharepoint.com/:x:/s/ReclaimCloud/EbbfYC1ejVxNkM0qbNguSLMBrqyG1o8uw88BsyQtEoDG0w?e=Lzvxwd   

20250916_Confidential - Carbon_Bombs_Companies_For D4G : https://reclaimfinance.sharepoint.com/:x:/s/ReclaimCloud/EUZhhddwz99KiqC5RK52LGUBJ9Pyo5ABsYfun5gqtXmH4g?e=t7TkD8



In [3]:
# Step 1
df_cb_emissions = load_rystad_emission_database(SHEETNAME_RYSTAD_CB_EMISSION)
df_expansion_emissions = load_rystad_emission_database(SHEETNAME_RYSTAD_EXPANSION_EMISSION)

# Step 2 
yearly_emissions_file_name = "20250718_Confidential - CarbonBombs_Yearly emissions.xlsx"
df_yearly_emissions = pd.read_excel(f'../data_sources/{yearly_emissions_file_name}', sheet_name='CB emissions', skiprows=2)
df_yearly_emissions = df_yearly_emissions.drop(columns=['Country'])
df_yearly_emissions = df_yearly_emissions.rename(columns={'Project': 'project_name_raw'})

# Step 3 
usecols_company = "A:I" 
company_file_name = "20250717_Confidential - Carbon_Bombs_Companies_For D4G.xlsx"
df_cb_companies = pd.read_excel(f'../data_sources/{company_file_name}', sheet_name='CB_CO2_detail', usecols=usecols_company)
df_expansion_companies = pd.read_excel(f'../data_sources/{company_file_name}', sheet_name='Expansion_Cies_CO2_detail', usecols=usecols_company)


# Step 4
bocc_file_name_v1 = "BOCC25_All FF_Aggregated_D4G.xlsx"
bocc_file_name_v2 = "Confidential_BOCC25_All FF_Aggregated_D4G with GOGEL ID.xlsx"
df_bocc = pd.read_excel(f'../data_sources/{bocc_file_name_v2}', sheet_name='BOCC_All FF_Aggregated_D4G')


# Load comparison between v1 and v2

In [4]:
def compare_carbon_bombs_version(cb_v1, cb_v2, gasoil_v2):
    # Clean project names when multiple country
    cb_v1["Project Name"] = cb_v1.apply(
        lambda row: row["Project Name"].removesuffix(f"_{row['Country']}") if isinstance(row["Project Name"], str) else row["Project Name"],
        axis=1
    )
    # Define manual matching dictionary
    manual_matching = {
        "Tarim (CNPC)": "Tarim",
        "Xinjiang (CNPC)": "Xinjiang",
        "Ahwaz Asmari":"Ahwaz (Ahwaz Asmari)",
        "Ahwaz Bangestan":"Ahwaz (Bangestan)",
        "South Pars (Phases 4-5) dry gas": "South Pars (Phases 4-5)",
        "Lula (X-Tupi)":"Tupi (x-Lula)",
        #"Longmaxi Shale (Sichuan/Changyu)": "Longmaxi Shale", # --> WARNING NOT IN NEW VERSION BECAUSE NOT THE SAME PROJECT
        "South Pars (Phases 2-3) dry gas":"South Pars (Phases 2-3)",
        "South Pars (Phases 9-10) dry gas":"South Pars (Phases 9-10)",
        "Tanzanian Coastal Offshore":"Tanzanian Coastal  Offshore", # With an extra space in new version
    }

    manual_matching_project = {}
    # First, create two dataframes with only the columns we want and rename them
    df_v1 = cb_v1[[
        'New_project',
        'Project Name',
        'Country',
        'Potential emissions (GtCO2)'
    ]].copy()
    df_v2 = cb_v2[[
        'project_name',
        'country',
        'producing_potential_emissions',
        'short_term_expansion_potential_emissions',
        'long_term_expansion_potential_emissions',
        'total_potential_emissions',
        'project_name_raw',
        'start_up_year',
        'latitude',
        'longitude'
    ]].copy()


    # Rename columns
    df_v1 = df_v1.rename(columns={
        'New_project': 'project_status_v1',
        'Project Name': 'project_name_v1',
        'Country': 'country_v1',
        'Potential emissions (GtCO2)': 'total_emissions_v1',
    })

    df_v2 = df_v2.rename(columns={
        'project_name': 'project_name_v2',
        'country': 'country_v2',
        'total_potential_emissions': 'total_emissions_v2',
        'producing_potential_emissions': 'producing_emissions_v2',
        'short_term_expansion_potential_emissions': 'short_term_emissions_v2',
        'long_term_expansion_potential_emissions': 'long_term_emissions_v2',
        'project_name_raw': 'project_name_raw_v2',
        'start_up_year': 'start_up_year_v2',
        'latitude': 'latitude_v2',
        'longitude': 'longitude_v2'
    })

    # Remove trailing and leading spaces from project names
    df_v1['project_name_v1'] = df_v1['project_name_v1'].str.strip()
    df_v2['project_name_v2'] = df_v2['project_name_v2'].str.strip()

    # Create a mapping series for manual matches
    manual_mapping = pd.Series(manual_matching)

    # Apply manual matching to project names while keeping the original country
    df_v1['matching_name'] = df_v1['project_name_v1'].map(manual_matching).fillna(df_v1['project_name_v1']).str.lower()
    df_v2['matching_name'] = df_v2['project_name_v2'].str.lower()

    # Create composite keys combining project name and country
    df_v1['matching_key'] = df_v1['matching_name'] + '|' + df_v1['country_v1'].str.lower()
    df_v2['matching_key'] = df_v2['matching_name'] + '|' + df_v2['country_v2'].str.lower()

    # Perform outer join using the composite key
    comparison_df = df_v1.merge(
        df_v2,
        on='matching_key',
        how='outer',
        indicator=True
    )

    # Check for country mismatches
    country_mismatches = []
    for idx, row in comparison_df.iterrows():
        if pd.notna(row['country_v1']) and pd.notna(row['country_v2']) and row['country_v1'] != row['country_v2']:
            LOGGER.warning(f"Country mismatch for project '{row['project_name_v1']}' -> '{row['project_name_v2']}': "
                        f"V1: {row['country_v1']} vs V2: {row['country_v2']}")
            country_mismatches.append({
                'project': row['project_name_v1'],
                'project_v2': row['project_name_v2'],
                'country_v1': row['country_v1'],
                'country_v2': row['country_v2']
            })

    # Clean up the DataFrame
    comparison_df = comparison_df.drop(['matching_name_x', 'matching_name_y'], axis=1, errors='ignore')

    # Print manual matches that were successful
    successful_manual_matches = comparison_df[
        comparison_df['project_name_v1'].isin(manual_matching.keys()) & 
        (comparison_df['_merge'] == 'both')
    ]
    if not successful_manual_matches.empty:
        print("\nSuccessful manual matches:")
        for _, row in successful_manual_matches.iterrows():
            print(f"V1: '{row['project_name_v1']}' -> V2: '{row['project_name_v2']}'")

    # Add distinction between methods (no match, with FID before 2050 or with no FID limit)
    # First, add a column matching_method that contains with no FID limit when there is a match
    comparison_df['matching_method'] = comparison_df.apply(
        lambda row: 'with no FID limit' if pd.notnull(row['project_name_v1']) and pd.notnull(row['project_name_v2']) 
        else 'no match',
        axis=1
    )

    # Filter rows from cb v1 where there is a no match
    unmatched_cb_v1 = comparison_df[
        comparison_df['project_name_v1'].notnull() &
        comparison_df['project_name_v2'].isnull()
    ]

    # Create a column matching_key in gasoil_v2 
    gasoil_v2['matching_key'] = gasoil_v2['project_name'].str.lower() + '|' + gasoil_v2['country'].str.lower()

    # Perform merge
    match_cb_v1_gasoil_projects = unmatched_cb_v1.merge(gasoil_v2, on='matching_key', how='left')

    # Update comparison df based on matched projects
    matched_keys = match_cb_v1_gasoil_projects['matching_key']
    matched_name_map = dict(zip(matched_keys, match_cb_v1_gasoil_projects['project_name']))
    mask = comparison_df['matching_key'].isin(matched_keys)
    comparison_df.loc[mask, 'matching_method'] = 'with FID before 2050'
    comparison_df.loc[mask, 'project_name_v2'] = comparison_df.loc[mask, 'matching_key'].map(matched_name_map)

    # Add manual matching for simple gasoil projects
    manual_matching_project = {}
    comparison_df.loc[comparison_df['project_name_v1'].isin(manual_matching_project.keys()), 'project_name_v2'] = comparison_df['project_name_v1'].map(manual_matching_project)
    comparison_df.loc[comparison_df['project_name_v1'].isin(manual_matching_project.keys()), 'matching_method'] = 'with FID before 2050'

    # Filter projects of V2 that are not Carbon Bombs (below 1GTCO2)
    comparison_df = comparison_df[~(comparison_df['project_name_v1'].isna() & (comparison_df['total_emissions_v2'] < 1))]

    # Calculate emissions difference (V2 - V1)
    comparison_df['total_emissions_difference'] = comparison_df['total_emissions_v2'] - comparison_df['total_emissions_v1']

    # Add percentage difference for emissions
    comparison_df['total_emissions_pct_difference'] = (
        (comparison_df['total_emissions_v2'] - comparison_df['total_emissions_v1']) / 
        comparison_df['total_emissions_v1'] * 100
    ).round(2)

    # 3. Define carbon bombs status changes
    conditions = [
        comparison_df['project_name_v1'].notna() & comparison_df['project_name_v2'].isna(),
        comparison_df['project_name_v1'].isna() & comparison_df['project_name_v2'].notna(),
        comparison_df['project_name_v1'].notna() & comparison_df['project_name_v2'].notna() & (comparison_df['total_emissions_v2'] < 1),
        comparison_df['project_name_v1'].notna() & comparison_df['project_name_v2'].notna() & (comparison_df['total_emissions_v2'] >= 1),
        comparison_df['project_name_v1'].notna() & comparison_df['project_name_v2'].notna()
    ]

    # Choices based on Nathan category for coal
    choices = [
        "project_not_found",
        "new_identified_carbon_bomb",
        "carbon_bomb_now_below_1gt",
        "carbon_bomb_above_1gt",
        "expansion_project"
    ]

    comparison_df['project_evolution'] = np.select(
        conditions, choices, default="Status unknown"
    )

    # 4. Flag has_started_since_v1
    comparison_df['has_started_since_v1'] = np.where(
        (comparison_df['project_status_v1'].fillna("") == "not started") &
        (comparison_df['producing_emissions_v2'].fillna(0) > 0),
        True, False
    )

    # Add GEM Link 
    # Carbon Bombs v1 output
    from carbon_bombs.conf import FPATH_OUT_CB
    cb_v1_with_gem_link = pd.read_csv(FPATH_OUT_CB)

    # Filter Oil&Gas projects
    cb_v1_with_gem_link = cb_v1_with_gem_link.loc[
        cb_v1_with_gem_link.Fuel_type_source_CB == 'Oil&Gas'
    ]

    # Filter out rows where project_name_v1 is not null
    comparison_df_with_only_v1 = comparison_df[comparison_df['project_name_v1'].notnull()].copy()

    # Sort both DataFrames
    comparison_df_sorted = comparison_df_with_only_v1.sort_values('total_emissions_v1').reset_index()
    cb_sorted = cb_v1_with_gem_link.sort_values('Potential_GtCO2_source_CB').reset_index(drop=True)

    # Security checks
    if len(comparison_df_sorted) != len(cb_sorted):
        raise ValueError("Mismatch in row counts between filtered comparison_df and cb_v1_with_gem_link.")
    if comparison_df_sorted['total_emissions_v1'].duplicated().any():
        raise ValueError("Duplicated values found in total_emissions_v1. Join by index may be unreliable.")

    # Perform index-based join
    comparison_df_sorted['GEM_url'] = cb_sorted['GEM_url_source_GEM'].values

    # Reassign the GEM_url values back to comparison_df_with_only_v1 using the original index
    comparison_df_with_only_v1 = comparison_df_sorted.set_index('index')
    comparison_df_with_only_v1.index.name = None  # Clean index name if needed

    # Update original comparison_df
    comparison_df = comparison_df.copy()
    comparison_df['GEM_url'] = None
    comparison_df.update(comparison_df_with_only_v1[['GEM_url']])

    # Clean up final DataFrame
    comparison_df = comparison_df.drop('_merge', axis=1)

    # Sort the DataFrame to group matched and unmatched projects
    comparison_df = comparison_df.sort_values(
        by=['project_name_v1', 'project_name_v2'],
        na_position='last'
    )


    # Reorder columns
    ordered_columns = [
        'project_name_v1',
        'country_v1',
        'project_status_v1',
        'total_emissions_v1',
        'project_name_v2',
        'country_v2',
        'latitude_v2',
        'longitude_v2',
        'start_up_year_v2',
        'producing_emissions_v2',
        'short_term_emissions_v2',
        'long_term_emissions_v2',
        'total_emissions_v2',
        'project_name_raw_v2',
        'total_emissions_difference',
        'project_evolution',
        'has_started_since_v1',
        'GEM_url'
    ]

    # Fix Collingham Shale, ZA name
    comparison_df.loc[comparison_df['project_name_raw_v2'] == ' Collingham Shale, ZA', 'project_name_raw_v2'] = 'Collingham Shale, ZA'

    # Reorder columns and save to CSV
    comparison_df = comparison_df[ordered_columns]

    # End of function return result
    return comparison_df

# Create file comparison between Carbon Bombs v1 and v2

In [5]:
cb_v1 = load_carbon_bomb_gasoil_database()
cb_emission_inferior_1gt = load_rystad_emission_database(SHEETNAME_RYSTAD_CB_EMISSION_INFERIOR_1GT)
# Remove project with column Total_potential_emissions_in_GTCO2 > 1GT in cb_emission_inferior_1gt
cb_emission_inferior_1gt = cb_emission_inferior_1gt[cb_emission_inferior_1gt["total_potential_emissions"] <= 1]
cb_v2 = pd.concat([df_cb_emissions,cb_emission_inferior_1gt])

df_cb_comparison = compare_carbon_bombs_version(cb_v1.copy(), cb_v2.copy(), df_expansion_emissions.copy())

# Coalesce project_name_v2 and project_name_v1 into project_name_v2 columns
df_cb_comparison['project_name_v2'] = df_cb_comparison['project_name_v2'].combine_first(df_cb_comparison['project_name_v1'])
df_cb_comparison['country_v2'] = df_cb_comparison['country_v2'].combine_first(df_cb_comparison['country_v1'])

# Rename columns name and filter columns from files comparison
renamed_comparison_columns = {
  'project_name_v2' : 'project_name',
  'country_v2' : 'country',
  'latitude_v2':'latitude',
  'longitude_v2':'longitude',
  'start_up_year_v2':'start_year',
  'project_status_v1':'project_status_in_v1',
  'total_emissions_v1': 'total_potential_emissions_v1',
  'total_emissions_v2': 'total_potential_emissions',
  'producing_emissions_v2': 'producing_potential_emissions',
  'short_term_emissions_v2': 'short_term_expansion_potential_emissions',
  'long_term_emissions_v2': 'long_term_expansion_potential_emissions',
  'project_name_raw_v2': 'project_name_raw',
  'project_evolution': 'project_evolution'
  
}

df_cb_comparison = df_cb_comparison.rename(columns=renamed_comparison_columns)
df_cb_comparison = df_cb_comparison[renamed_comparison_columns.values()]

# Create a copy into df_cb_emissions
df_cb_emissions = df_cb_comparison.copy()



Successful manual matches:
V1: 'Ahwaz Asmari' -> V2: 'Ahwaz (Ahwaz Asmari)'
V1: 'Ahwaz Bangestan' -> V2: 'Ahwaz (Bangestan)'
V1: 'South Pars (Phases 2-3) dry gas' -> V2: 'South Pars (Phases 2-3)'
V1: 'South Pars (Phases 4-5) dry gas' -> V2: 'South Pars (Phases 4-5)'
V1: 'South Pars (Phases 9-10) dry gas' -> V2: 'South Pars (Phases 9-10)'
V1: 'Tanzanian Coastal Offshore' -> V2: 'Tanzanian Coastal  Offshore'
V1: 'Tarim (CNPC)' -> V2: 'Tarim'
V1: 'Lula (X-Tupi)' -> V2: 'Tupi (x-Lula)'
V1: 'Xinjiang (CNPC)' -> V2: 'Xinjiang'


# Rename columns and replace Country name for companies

In [6]:
# Carbon bombs companies 
renamed_columns_cb_companies = {
    "Project": "project",
    "Country": "country",
    "Company": "company",
    "GOGEL ID": "gogel_id",
    "Headquarter Country": "headquarter_country",
    "Company ISIN": "company_isin",
    "Company LEI": "company_lei",
    "Total potential emissions (mmtCO2)": "potential_emissions_mmtco2",
    "Total potential emissions (gtCO2)2": "potential_emissions_gtco2",
}

renamed_columns_expansion_companies = {
    "Project": "project",
    "Country": "country",
    "Company": "company",
    "GOGEL ID": "gogel_id",
    "Headquarter Country": "headquarter_country",
    "Company ISIN": "company_isin",
    "Company LEI": "company_lei",
    "Potential emissions (mmtCO2)": "potential_emissions_mmtco2",
    "Potential emissions (gtCO2)": "potential_emissions_gtco2",
}

# Clean country for some companies 
country_updates = {
    "Blue Sky Resources Ltd": "Canada",
    "Petroleum Sarawak Bhd (PETROS)": "Malaysia",
    "Energia Argentina SA": "Argentine",
    "Confluence Resources LP": "United States"
}
for company, country in country_updates.items():
    df_expansion_companies.loc[df_expansion_companies['Company'] == company, 'Headquarter Country'] = country
for company, country in country_updates.items():
    df_cb_companies.loc[df_cb_companies['Company'] == company, 'Headquarter Country'] = country


df_cb_companies.rename(columns=renamed_columns_cb_companies, inplace=True) 
df_expansion_companies.rename(columns=renamed_columns_expansion_companies, inplace=True)

# Replace UAE by United Arab Emirates in country column and headquarter_country column
df_cb_companies["country"] = df_cb_companies["country"].replace("UAE", "United Arab Emirates")
df_cb_companies["headquarter_country"] = df_cb_companies["headquarter_country"].replace("UAE", "United Arab Emirates")
df_expansion_companies["country"] = df_expansion_companies["country"].replace("UAE", "United Arab Emirates")
df_expansion_companies["headquarter_country"] = df_expansion_companies["headquarter_country"].replace("UAE", "United Arab Emirates")

# Replace Timor Sea JPDA by Timor-Leste in headquarter_country column
df_cb_companies["headquarter_country"] = df_cb_companies["headquarter_country"].replace("Timor Sea JPDA", "Timor-Leste")
df_expansion_companies["headquarter_country"] = df_expansion_companies["headquarter_country"].replace("Timor Sea JPDA", "Timor-Leste") 

# Remap project values 
manual_project_mapping = {
    # Project name in df_companies_involvement : Project name in df_gasoil_emissions
    'Athabasca Oil Sands, CA': 'Athabasca Oil Sands Project, CA',
    'Qatargas 1 LNG T1-T3, QA': 'QatarGas 1 LNG T1-T3, QA',
    'Qatargas LNG T12-T13 (NFE-South), QA': 'QatarGas LNG T12-T13 (NFE-South), QA',
    'Qatargas LNG T8-T11 (NFE-East), QA':'QatarGas LNG T8-T11 (NFE-East), QA',
    'Rasgas 2 (RL 2) LNG T3-T5, QA':'Rasgas 2 LNG T3-T5, QA',
    'Rasgas 3 (RL 3) LNG T6-T7, QA':'Rasgas 3 LNG T6-T7, QA',
    'Gazprom dobycha Noyabrsk, RU':'Gazprom Dobycha Noyabrsk, RU',
    'Qatargas LNG T14-T15 (NFE-West), QA':'QatarGas LNG T14-T15 (NFE-West), QA'
}
df_cb_companies['project'] = df_cb_companies['project'].replace(manual_project_mapping)
df_expansion_companies['project'] = df_expansion_companies['project'].replace(manual_project_mapping)
    

# Cleaning and merge Carbon Bombs and Expansion files

In [7]:
# Step 1 : Convert Expansion emission in mtCo2 

columns_to_convert_in_GtCO2 = [
    "total_potential_emissions",
    "producing_potential_emissions",
    "short_term_expansion_potential_emissions",
    "long_term_expansion_potential_emissions"
]
df_expansion_emissions[columns_to_convert_in_GtCO2] = df_expansion_emissions[columns_to_convert_in_GtCO2] / 1000

# Step 2 : Remove carbon_bombs projects from expansion dataset to avoid duplicates after merge (UNION)

# Before removing those projects, we define a list of projects that are present in both datasets
# to filter them out from expansion companies connexion 
cb_in_expansion = df_cb_emissions[df_cb_emissions["project_name_raw"].isin(df_expansion_emissions["project_name_raw"])]
cb_in_expansion_list = cb_in_expansion["project_name_raw"].tolist()

# Add column new_extraction_after_2021 based of cb presence in expansion sheet
df_expansion_emissions["new_extraction_after_2021"] = "True"
df_cb_emissions["new_extraction_after_2021"] = df_cb_emissions["project_name_raw"].isin(cb_in_expansion_list).map({True: "True", False: "False"})

# Add a column percentage_emissions_new_extraction
def calc_percentage_emissions_new_extraction(row):
    if row["project_name_raw"] in cb_in_expansion_list:
        # Get the total_potential_emissions for this project in expansion dataset
        expansion_value = df_expansion_emissions.loc[
            df_expansion_emissions["project_name_raw"] == row["project_name_raw"], 
            "total_potential_emissions"
        ]
        # Calculate percentage compared to CB emissions
        cb_value = row["total_potential_emissions"]
        # Avoid division by zero
        if cb_value == 0:
            return ""
        percentage = (expansion_value / cb_value) * 100
        return f"{float(percentage):.1f}%"
    else:
        return ""
    
df_cb_emissions["percentage_emissions_new_extraction"] = df_cb_emissions.apply(calc_percentage_emissions_new_extraction, axis=1)
df_expansion_emissions["percentage_emissions_new_extraction"] = ""


# To list those project of CB not in Expansion use following code :
cb_not_in_expansion = df_cb_emissions[~df_cb_emissions["project_name_raw"].isin(df_expansion_emissions["project_name_raw"])]
print("Total projects in CB that are NOT in expansion (won't be removed):", len(cb_not_in_expansion))

print(f"Projects in expansion before filtering: {df_expansion_emissions.shape}")
df_expansion_emissions = df_expansion_emissions[~df_expansion_emissions["project_name_raw"].isin(df_cb_emissions["project_name_raw"])]
print(f"Projects in expansion after filtering: {df_expansion_emissions.shape}")

# Step 3 : Add columns project_type
df_expansion_emissions["project_type"] = "New extraction projects"
df_cb_emissions["project_type"] = "Carbon bombs"

# Step 3 : Add column project_status_in_v1 for df_expansion_emissions
# Column is already present in df_cb_emissions with values not_started and operating
df_expansion_emissions["project_status_in_v1"] = ""
df_cb_emissions.loc[df_cb_emissions["project_evolution"]=="new_identified_carbon_bomb","project_status_in_v1"] = ""

# Step 3 : Fulfill project_evolution for expansion projects
df_expansion_emissions["project_evolution"] = "new_extraction_projects"

# Step 3 ter : Rename column start_up_year to start_year
df_expansion_emissions.rename(columns={"start_up_year": "start_year"}, inplace=True)

# Step 4 : Union datasets df_cb_emissions and df_expansion_emissions

df_gasoil_emissions = pd.concat([df_cb_emissions, df_expansion_emissions], ignore_index=True)
print(f"Total projects in gasoil after union: {df_gasoil_emissions.shape}")

# Step 5 : Convert start_year to int
df_gasoil_emissions["start_year"] = df_gasoil_emissions["start_year"].astype('Int64')

# Step 6 : Add status based on operating short term and log term emissions 
custom_project_status_dict = {
    'Kronprins Christian Offshore , GL': 'stopped',
    'Barail Shale, IN': 'stopped'
}
def determine_project_status(row):
    if row['project_name_raw'] in custom_project_status_dict:
        return custom_project_status_dict[row['project_name_raw']]

    producing = row['producing_potential_emissions'] > 0
    short_term = row['short_term_expansion_potential_emissions'] > 0
    long_term = row['long_term_expansion_potential_emissions'] > 0

    if producing and not short_term and not long_term:
        return "operating"
    elif producing and (short_term or long_term):
        return "operating and expanding"
    elif (short_term or long_term) and not producing:
        return "not started"
    else:
        return "project not found" 

df_gasoil_emissions['project_status'] = df_gasoil_emissions.apply(determine_project_status, axis=1)

# Step 7 : Add world region based on country with mapping only for unique countries
unique_countries = df_gasoil_emissions["country"].unique()
country_to_region = {country: get_world_region(country) for country in unique_countries}
df_gasoil_emissions["world_region"] = df_gasoil_emissions["country"].map(country_to_region)

# Step 8 : Add column fuel_type
df_gasoil_emissions["fuel_type"] = "Oil & Gas"

# Step 9 : Round emissions columns 
rounding_cols = [
    "producing_potential_emissions",
    "short_term_expansion_potential_emissions",
    "long_term_expansion_potential_emissions"
]
df_gasoil_emissions[rounding_cols] = df_gasoil_emissions[rounding_cols].round(3)
df_gasoil_emissions["total_potential_emissions"] = df_gasoil_emissions[rounding_cols].sum(axis=1)
df_gasoil_emissions["total_potential_emissions_v1"] = df_gasoil_emissions["total_potential_emissions_v1"].round(3)

# Step 10 Calculate percentage_evolution between v1 & v2 
df_gasoil_emissions['percentage_evolution'] = (100*(df_gasoil_emissions['total_potential_emissions'] - df_gasoil_emissions['total_potential_emissions_v1']) / df_gasoil_emissions['total_potential_emissions_v1']).round(1).astype(str).str.replace('.', ',', regex=False) + '%'
df_gasoil_emissions['percentage_evolution'] = df_gasoil_emissions['percentage_evolution'].replace('nan%', '')

# Step 11 Add latitude and longitude for carbon bombs v1 not matched (source GEM)
df_v1_latitude = pd.read_excel(
    f'../data_cleaned/carbon_bombs_all_datasets.xlsx',
    sheet_name='carbon_bombs_data',
    usecols = ['Carbon_bomb_name_source_CB', 'Latitude', 'Longitude']
    )

# Filter rows with missing latitude or longitude
mask_missing_coordinates = df_gasoil_emissions["latitude"].isna() | df_gasoil_emissions["longitude"].isna()
df_missing = df_gasoil_emissions.loc[mask_missing_coordinates]

# For 7 projects of cb_v1 not matched with cb_v2 we don't have duplicate project name
# Check for duplicates in project_name
if df_missing["project_name"].duplicated().any():
    print("⚠️ Warning: df_missing has duplicate project_name values")
df_missing = df_missing.merge(
    df_v1_latitude,
    left_on="project_name",
    right_on="Carbon_bomb_name_source_CB",
    how="left"
)

# Fill missing latitude/longitude
df_missing["latitude"] = df_missing["latitude"].fillna(df_missing["Latitude"])
df_missing["longitude"] = df_missing["longitude"].fillna(df_missing["Longitude"])

# Create mapping dictionaries
lat_map = df_missing.set_index("project_name")["latitude"].to_dict()
lon_map = df_missing.set_index("project_name")["longitude"].to_dict()

# Fill missing values in df_gasoil_emissions
df_gasoil_emissions["latitude"] = df_gasoil_emissions["latitude"].fillna(df_gasoil_emissions["project_name"].map(lat_map))
df_gasoil_emissions["longitude"] = df_gasoil_emissions["longitude"].fillna(df_gasoil_emissions["project_name"].map(lon_map))


# Step 12 : Round latitude and longitude columns with 3 decimals (~100m)
df_gasoil_emissions[["latitude","longitude"]] = df_gasoil_emissions[["latitude","longitude"]].round(2)


Total projects in CB that are NOT in expansion (won't be removed): 113
Projects in expansion before filtering: (2092, 12)
Projects in expansion after filtering: (1979, 12)
Total projects in gasoil after union: (2205, 16)


/var/folders/t3/myv5sjjd0537bvgr90_h_fcc0000gn/T/ipykernel_32860/3383551155.py:36: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  return f"{float(percentage):.1f}%"


# Create company data sheet related to Gasoil projects (Carbon bombs and Expansion)


In [8]:
# Step 1 : Create company df to only store compnay informations
df_companies_data = pd.concat([df_cb_companies , df_expansion_companies], ignore_index=True)

# Step 2 : Remove columns related to emissions 
df_companies_data = df_companies_data.drop(columns=["project","country","potential_emissions_mmtco2","potential_emissions_gtco2"])

# Step 3 : Remove company with empty gogel_id
df_companies_data = df_companies_data[df_companies_data["gogel_id"].notna()].reset_index(drop=True)

# Step 4 : Convert gogel_id to int
df_companies_data["gogel_id"] = df_companies_data["gogel_id"].astype(int)

# Step 5 : Remove duplicates
df_companies_data = df_companies_data.drop_duplicates(subset=["company","headquarter_country"]).reset_index(drop=True)

# Step 6 : Add latitude and longitude based on country headquarters
df_companies_data = _add_country_lat_long(df_companies_data, country_col="headquarter_country")

# Step 7 : Add world region based on country with mapping only for unique countries
unique_countries = df_companies_data["headquarter_country"].unique()
country_to_region = {country: get_world_region(country) for country in unique_countries}
df_companies_data["world_region"] = df_companies_data["headquarter_country"].map(country_to_region)

# Step 8 : Reorder column order
df_companies_data = df_companies_data[["company","headquarter_country","latitude","longitude","world_region","gogel_id","company_isin","company_lei"]]



# Create connexion project company sheet related to Gasoil projects (Carbon bombs and Expansion)


In [9]:
# Step 0 : Convert gogel_id to int
df_expansion_companies["gogel_id"] = df_expansion_companies["gogel_id"].astype("Int64")
df_cb_companies["gogel_id"] = df_cb_companies["gogel_id"].astype("Int64")

# Step 1 : Remove cb_in_expansion_list project from df_expansion_companies to avoid duplicates
df_expansion_companies = df_expansion_companies[~df_expansion_companies["project"].isin(cb_in_expansion_list)]
# Step 1 bis : Remove project from df_cb_companies if project not present in df_cb_emissions
df_cb_companies = df_cb_companies[df_cb_companies["project"].isin(df_cb_emissions["project_name_raw"])]
# Step 1 ter : Remove project from df_expansion_companies if project present in df_cb_emissions
df_expansion_companies = df_expansion_companies[~df_expansion_companies["project"].isin(df_cb_emissions["project_name_raw"])]


# Step 2 : Filter country acronym from project column and create project_name_raw column
clean_project_names_with_iso(df_cb_companies, column_name="project")
clean_project_names_with_iso(df_expansion_companies, column_name="project")

# Step 3 : Calculate emissions percentage interval for company involvment to anonymize data
# Step 3.1 : Calculate percentage of total emissions for each company
def get_involvement_interval(pct):
    if pd.isna(pct):
        return "Unknown" 
    if pct <= 0:
        return "0%"
    elif pct <= 5:
        return "0-5%"
    elif pct <= 10:
        return "5-10%"
    else:
        # For anything above 10%, use 10% bins
        lower = int(pct // 10) * 10
        upper = lower + 10
        if upper > 100:
            return f"{lower}%"
        return f"{lower}-{upper}%"

def add_involvement_percentage(df, column_total_emissions):
    df['project_total_emissions'] = df.groupby('project_name_raw')[column_total_emissions].transform('sum')
    df['company_involvement_percent'] = df[column_total_emissions] / df['project_total_emissions'] * 100
    return df

df_cb_companies = add_involvement_percentage(df_cb_companies, "potential_emissions_mmtco2")
df_expansion_companies = add_involvement_percentage(df_expansion_companies, "potential_emissions_mmtco2")

# For informations to Lou : Calculate sums of potential_emissions_mmtco2 related to companies 
# that do not have gogel_id and project is not equal to Open Acreage
# For cb_companies
filtered_df_cb_companies = df_cb_companies[
    df_cb_companies['gogel_id'].isna() &
    (df_cb_companies['project'] != "Open acreage")
]
total_emissions_with_no_gogel_id = filtered_df_cb_companies['potential_emissions_mmtco2'].sum()
total_emissions = df_cb_companies['potential_emissions_mmtco2'].sum()
print("Total carbon bombs emissions with no gogel_id : ", total_emissions_with_no_gogel_id)
print("Total carbon bombs emissions : ", total_emissions)

# For expansion companies
filtered_df_expansion_companies = df_expansion_companies[
    df_expansion_companies['gogel_id'].isna() &
    (df_expansion_companies['project'] != "Open acreage")
]
total_emissions_with_no_gogel_id = filtered_df_expansion_companies['potential_emissions_mmtco2'].sum()
total_emissions = df_expansion_companies['potential_emissions_mmtco2'].sum()
print("Total expansion emissions with no gogel_id : ", total_emissions_with_no_gogel_id)
print("Total expansion emissions : ", total_emissions)

# Step 3.2 : Replace company by Others for companies that do not have gogel_id and is not equal 'Open acreage'
def clean_and_group_companies(df):
    # Step 1: Drop unused columns
    columns_to_drop = [
        'headquarter_country',
        'company_isin',
        'company_lei',
        'potential_emissions_mmtco2',
        'project_total_emissions'
    ]
    df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

    # Step 2: Replace company name by 'Others' if gogel_id is missing and not 'Open acreage'
    mask = df['gogel_id'].isna() & (df['company'] != 'Open acreage')
    df.loc[mask, 'company'] = 'Others'

    # Step 3: Separate 'Others' and group by project_name_raw
    df_others = df[df['company'] == 'Others']
    df_non_others = df[df['company'] != 'Others']

    df_others_grouped = df_others.groupby(
        ['project_name_raw', 'company'], as_index=False
    )['company_involvement_percent'].sum()

    # Step 4: Combine back and sort
    df_cleaned = pd.concat([df_others_grouped, df_non_others], ignore_index=True)
    df_cleaned = df_cleaned.sort_values(by=['project_name_raw', 'company']).reset_index(drop=True)

    return df_cleaned

df_cb_companies = clean_and_group_companies(df_cb_companies)
df_expansion_companies = clean_and_group_companies(df_expansion_companies)

# More analysis for Lou
# Objective know how many projects have company Others that reprensents more than 30% share
def analyze_others_involvement(df, df_name=""):
    total_projects = df["project_name_raw"].nunique()

    projects_with_others_gt_30 = df[
        (df["company"] == "Others") &
        (df["company_involvement_percent"] > 30)
    ]["project_name_raw"].nunique()

    print(f"\n--- {df_name} ---")
    print(f"Total number of projects: {total_projects}")
    print(f"Projects with 'Others' > 30%: {projects_with_others_gt_30}")

# Run the analysis on both datasets
analyze_others_involvement(df_cb_companies, df_name="Carbon Bombs")
analyze_others_involvement(df_expansion_companies, df_name="Expansion")

# Step 3.3 : Add company_involvement_percent interval
df_cb_companies['involvement_interval'] = df_cb_companies['company_involvement_percent'].apply(get_involvement_interval)
df_expansion_companies['involvement_interval'] = df_expansion_companies['company_involvement_percent'].apply(get_involvement_interval)

# Step 3.4 : Drop useless columns 
columns_to_drop = [
    'project',
    'country',
    'potential_emissions_gtco2'
]
df_cb_companies = df_cb_companies.drop(columns=columns_to_drop)
df_expansion_companies = df_expansion_companies.drop(columns=columns_to_drop)

# Step 4 : Add columns gasoil_type to differenciate and concat dataframe
df_cb_companies["project_type"] = "Carbon bombs"
df_expansion_companies["project_type"] = "New extraction projects"
df_companies_involvement = pd.concat([df_cb_companies, df_expansion_companies], ignore_index=True)

# Step 5 : Add columns fuel_type = Oil&Gas
df_companies_involvement['fuel_type'] = 'Oil&Gas'



Total carbon bombs emissions with no gogel_id :  106091.408963673
Total carbon bombs emissions :  455821.6018026953
Total expansion emissions with no gogel_id :  30508.27592151607
Total expansion emissions :  97695.58742967395

--- Carbon Bombs ---
Total number of projects: 219
Projects with 'Others' > 30%: 7

--- Expansion ---
Total number of projects: 2089
Projects with 'Others' > 30%: 263


# Add column listing company involved in carbon bombs project

In [10]:

def get_lower_bound(interval):
    if interval == '100%':
        return 100
    # for intervals like "90-100%", split and take first number
    else:
        try:
            return int(interval.split('-')[0])
        except:
            return np.nan  # in case of unexpected format
            
# Create a numeric column to sort by
df_companies_involvement['interval_lower_bound'] = df_companies_involvement['involvement_interval'].apply(get_lower_bound)

# Sort by project and the numeric lower bound
df_companies_involvement_sorted = df_companies_involvement.sort_values(['project_name_raw', 'interval_lower_bound'], ascending=[True, False])

# Group and concatenate
concat_company_involvement_in_project = (
    df_companies_involvement_sorted
    .groupby('project_name_raw')[['company', 'involvement_interval']]
    .apply(lambda x: ' | '.join(x['company'] + ' (' + x['involvement_interval'] + ')'))
    .reset_index()
    .rename(columns={0: 'list_of_company_involved'})
)

# Drop the temporary column
df_companies_involvement.drop(columns=['interval_lower_bound'], inplace=True)

# Merge back
df_gasoil_emissions = df_gasoil_emissions.merge(concat_company_involvement_in_project, on='project_name_raw', how='left')

# Filter relationship between project and connexion_project_company
set_project = set(df_gasoil_emissions['project_name_raw'].unique())
df_companies_involvement = df_companies_involvement[df_companies_involvement['project_name_raw'].isin(set_project)]

# Add yearly emissions for carbon bombs project between 2020 and 2025

In [11]:
renamed_yearly_emissions = {
    2020: "2020_estimated_emissions",
    2021: "2021_estimated_emissions",
    2022: "2022_estimated_emissions",
    2023: "2023_estimated_emissions",
    2024: "2024_estimated_emissions",
    2025: "2025_estimated_emissions",
}

replace_project_name = {
    "Gazprom dobycha Noyabrsk, RU":"Gazprom Dobycha Noyabrsk, RU",
    "Qatargas LNG T12-T13 (NFE-South), QA":"QatarGas LNG T12-T13 (NFE-South), QA",
    "Qatargas 1 LNG T1-T3, QA":"QatarGas 1 LNG T1-T3, QA",
    "Qatargas LNG T8-T11 (NFE-East), QA":"QatarGas LNG T8-T11 (NFE-East), QA",
    "Qatargas LNG T14-T15 (NFE-West), QA":"QatarGas LNG T14-T15 (NFE-West), QA",
    "Rasgas 2 (RL 2) LNG T3-T5, QA":"Rasgas 2 LNG T3-T5, QA",
    "Rasgas 3 (RL 3) LNG T6-T7, QA":"Rasgas 3 LNG T6-T7, QA"
}

df_yearly_emissions["project_name_raw"] = df_yearly_emissions["project_name_raw"].replace(replace_project_name)


df_yearly_emissions = df_yearly_emissions.rename(columns=renamed_yearly_emissions)

df_yearly_subset = df_yearly_emissions[
    ['project_name_raw'] + list(renamed_yearly_emissions.values())
].copy()

df_yearly_subset['2020_2025_past_emissions'] = (
    df_yearly_subset[list(renamed_yearly_emissions.values())].sum(axis=1)
)

df_gasoil_emissions = df_gasoil_emissions.merge(
    df_yearly_subset,
    on='project_name_raw',
    how='left'
)

# Delete value for column 2020_2025_past_emissions for expansion projects
columns_to_clean = [
    "2020_estimated_emissions",
    "2021_estimated_emissions", 
    "2022_estimated_emissions", 
    "2023_estimated_emissions", 
    "2024_estimated_emissions", 
    "2025_estimated_emissions",
    "2020_2025_past_emissions"
]
df_gasoil_emissions.loc[df_gasoil_emissions['project_evolution'] == "new_extraction_projects",columns_to_clean] = np.nan

# Create LNG sheet and Country Data sheet



In [12]:
# STEP 1 : LNG
# Last version of the file given here : https://reclaimfinance.sharepoint.com/:x:/s/ReclaimCloud/EVjOiwKFSk1LvBBKzHVLvTQBI0pHdYkHQjOYLIYLt7oDwA?rtime=xdbtGPH_3Ug
from carbon_bombs.processing.lng import create_lng_table
df_lng = create_lng_table()

# STEP 2 : GogelID for LNG 
df_company_gogel_id = pd.read_csv(f"{DATA_SOURCE_PATH}/gogel_2024_all_companies.csv", sep = ";")
df_company_gogel_id.head()

df_lng_expanded = (
    df_lng
    .assign(company=df_lng['companies_involved'].str.split(','))
    .explode('company')
)

# Step 2: clean company names
df_lng_expanded['company'] = df_lng_expanded['company'].str.strip()

# Step 3: prepare a normalized key for merge (case-insensitive)
df_lng_expanded['company_norm'] = df_lng_expanded['company'].str.lower().str.strip()
df_company_gogel_id['company_norm'] = df_company_gogel_id['company'].str.lower().str.strip()

# Step 4: merge
df_lng_companies = df_lng_expanded.merge(
    df_company_gogel_id[['company_norm', 'gogel_id']],
    on='company_norm',
    how='left'
)

# Step 5: keep only required columns
df_lng_companies = df_lng_companies[['project_name', 'company', 'gogel_id']]

# Step 6 : Add columns involvement_interval, project_type, fuel_type 
# to concat with df_companies_involvement
df_lng_companies['involvement_interval'] = ""
df_lng_companies['project_type'] = "New LNG terminal"
df_lng_companies['fuel_type'] = ""

# Step 7 : Rename columns before concatenation
df_companies_involvement = df_companies_involvement.rename(
    columns={"project_name_raw": "project_name"}
)

# Step 8 : Concatenate companies involment for Oil&Gas and LNG
df_companies_involvement = pd.concat([df_companies_involvement, df_lng_companies], ignore_index=True)



In [13]:
# STEP 2 : Country
from carbon_bombs.io.undata import load_undata

def get_countries(df):
    df = df[["country"]]
    # Split mulitple countries, separated by '-', in multiple countries
    df = df.assign(country=df["country"].str.split("-")).explode("country")
    # Remove duplicates
    df = df.drop_duplicates()
    # Sort by increasing values
    df = df.sort_values(by="country", ascending=True)
    df.reset_index(drop=True, inplace=True)
    return df

def format_serie_values(val):
    """Format string as interger or float"""
    if not isinstance(val, str):
        return val

    val = val.replace(",", "")

    if "." in val:
        return float(val)

    return int(val)


def format_countries_df_with_wanted_series(df_countries: pd.DataFrame) -> pd.DataFrame:
    columns_map = {
        "Population mid-year estimates (millions)": "Population_in_millions",
        "Surface area (thousand km2)": "Surface_thousand_km2",
        "GDP in current prices (millions of US dollars)": "GDP_millions_US_dollars",
        "GDP per capita (US dollars)": "GDP_per_capita_US_dollars",
        "Emissions (thousand metric tons of carbon dioxide)": "Emissions_thousand_tons_CO2",
        "Emissions per capita (metric tons of carbon dioxide)": "Emissions_per_capita_tons_CO2",
    }

    # Keep only some wanted KPI
    df_countries_filtered = df_countries.loc[
        df_countries["Series"].isin(columns_map.keys())
    ]
    # get max years by country and serie
    country_year_max = df_countries_filtered.groupby(["country", "Series"]).agg(
        year_max=("Year", "max")
    )
    country_year_max_df = country_year_max.merge(
        df_countries,
        left_on=["country", "Series", "year_max"],
        right_on=["country", "Series", "Year"],
    ).drop(columns=["year_max"])

    # Change serie name to wanted format
    country_year_max_df["Series"] = country_year_max_df["Series"].replace(columns_map)

    # Init final countries df
    final_countries_df = df_countries[["country"]].drop_duplicates()

    for serie, serie_df in country_year_max_df.groupby("Series"):
        # Pivot dataframe to get values and year into the same row
        serie_df = serie_df.pivot(
            index=["country"], columns=["Series"], values=["Value", "Year"]
        ).reset_index()

        # Format year dtype
        serie_df["Year"] = serie_df["Year"].astype(int)

        # Change columns name
        serie_df.columns = ["country", serie, f"Year_{serie}"]

        # Format values
        serie_df[serie] = serie_df[serie].apply(format_serie_values)

        # merge to add KPI for each countries with the last year available for this metric
        final_countries_df = final_countries_df.merge(serie_df, on=["country"])

    return final_countries_df

df_gasoil_countries = get_countries(df_gasoil_emissions)
df_undata = load_undata()

df_countries = df_gasoil_countries.merge(
    df_undata,
    left_on="country",
    right_on="Region_Country_Area_name",
    how="inner",
    sort=True,
)

df_countries = format_countries_df_with_wanted_series(df_countries)


# Create bank_information table

In [14]:
"""Function to process banks information"""
import pandas as pd

from carbon_bombs.io.banktracks import scrapping_description_bank_page
from carbon_bombs.io.banktracks import scrapping_main_page_bank_track
from carbon_bombs.utils.location import get_world_region
from carbon_bombs.utils.logger import LOGGER


def process_raw_info(dict_info):
    """
    Process a dictionary of raw information and return a cleaned dictionary.

    Parameters
    ----------
    dict_info: dict
        A dictionary containing raw information.

    Returns
    -------
    dict:
        A dictionary containing cleaned information.

    Examples
    --------
    >>> dict_info = {"Website": "<a href='https://www.example.com'>Example</a>"}
    >>> process_raw_info(dict_info)
    {'Bank Website': 'Example'}

    """
    # Instanciate clean_dict that will contains info extracted from raw
    clean_dict = {
        "Bank Website": "None",
        "Headquarters address": "None",
        "Headquarters country": "None",
        "CEO Name": "None",
        "Board description": "None",
        "Supervisor Name": "None",
        "Supervisor Website": "None",
        "Shareholder structure source": "None",
    }

    if dict_info["Website"] != "None":
        clean_dict["Bank Website"] = dict_info["Website"].find("a").text

    if dict_info["Headquarters"] != "None":
        # Example for full_address
        # [<div>Gustav Mahlerlaan 10</div>, <div> 1082 PP Amsterdam</div>, <div>Netherlands</div>]
        full_address = dict_info["Headquarters"].find_all("div")

        address = f"{full_address[0].text.strip()},{full_address[1].text.strip()}"
        country = full_address[-1].text.strip()

        clean_dict["Headquarters address"] = address
        clean_dict["Headquarters country"] = country

    if dict_info["CEO/chair"] != "None":
        a_tag = dict_info["CEO/chair"].find("a")
        # if a_tag not empty
        if a_tag:
            # Extract the URL, see an example:
            # <a href="http://www.abnamro.nl/en/index.html" target="_blank">http://www.abnamro.nl/en/index.html</a>
            url_address = a_tag["href"]
            # Extract the CEO's name
            name_ceo = a_tag.text

            clean_dict["CEO Name"] = name_ceo
            clean_dict["Board description"] = url_address

    if dict_info["Supervisor"] != "None":
        # example of a wanted anchor
        # <a href="http://www.rba.gov.au/" target="_blank">Reserve Bank of Australia</a>
        anchor = dict_info["Supervisor"].find("a")

        if anchor and not isinstance(anchor, int):
            clean_dict["Supervisor Name"] = anchor.text
            clean_dict["Supervisor Website"] = anchor["href"]

    if dict_info["Ownership"] != "None":
        url = dict_info["Ownership"].find("a")
        # if url not empty
        if url:
            clean_dict["Shareholder structure source"] = url["href"]

    # Return cleaned dictionary
    return clean_dict


def create_banks_table(df_bocc):
    """
    Create a Pandas DataFrame from scraped information and return it.

    Args:
        url (str): The URL of the main page to scrape.

    Returns:
        pandas.DataFrame: A Pandas DataFrame containing information on bank
        companies.

    Example:
        >>> url = "https://example.com"
        >>> create_bank_dataframe(url)
            Bank Name  Bank Website  Headquarters address  Headquarters country
        0    Example Bank www.example.com 123 Main St, Anytown  USA
        1    Another Bank www.anotherbank.com 456 Oak St, Anycity  USA
        ...
    """
    LOGGER.debug("Start creation of banks dataset")
    LOGGER.debug("Get banks name connected to companies")
    cnx_bank_comp = df_bocc.copy()
    # Make a remap of bank name based on manual_match_bank in order to have
    # coherent key values in BOCC and banking_informations.csv
    manual_match_bank = {
        'La Caixa Group':'CaixaBank',
        'US Bancorp':'U.S. Bancorp',
        'Truist Financial':'Truist Financial Corporation',
        'NatWest':'NatWest Group',
        'Ping An Insurance Group':'Ping An Bank',
        'BMO Financial Group':'Bank of Montreal (BMO)',
        'Groupe BPCE':'BPCE',
        'CIBC':'Canadian Imperial Bank of Commerce (CIBC)',
        'Itaú Unibanco':'Itaú-Unibanco',
        'CITIC':'CITIC Bank International',
        'China Minsheng Banking':'China Minsheng Bank',
        'Toronto-Dominion Bank':'Toronto-Dominion Bank (TD Bank)',
        'Industrial and Commercial Bank of China':'Industrial and Commercial Bank of China (ICBC)',
        'Citigroup':'Citi',
        'Mizuho Financial':'Mizuho Financial Group',
        'National Australia Bank':'National Australia Bank (NAB)',
        'Mitsubishi UFJ Financial':'Mitsubishi UFJ Financial Group (MUFG)',
        'Royal Bank of Canada':'Royal Bank of Canada (RBC)',
        'Industrial Bank Company':'Industrial Bank',
        'Banco Bilbao Vizcaya Argentaria (BBVA)':'BBVA',
        'China Everbright':'China Everbright Bank',
        'ING Group':'ING',
        'Santander':'Banco Santander',
        'PNC Financial Services':'PNC',
        'Commonwealth Bank of Australia':'Commonwealth Bank',
        'SMBC Group':'Sumitomo Mitsui Financial Group',
        #'Capital One Financial':,
    }
    cnx_bank_comp["Bank"] = cnx_bank_comp["Bank"].replace(manual_match_bank)
    banks_in_bocc = cnx_bank_comp["Bank"].unique()
    
    # Instanciate dataframe containing srapped info
    columns_dataframe = [
        "Bank Name",
        "Bank Website",
        "Headquarters address",
        "Headquarters country",
        "CEO Name",
        "Board description",
        "Supervisor Name",
        "Supervisor Website",
        "Shareholder structure source",
        "Source BankTrack",
        "Latitude",
        "Longitude",
    ]
    df = pd.DataFrame(columns=columns_dataframe)
    LOGGER.debug("Scrap all banks from banktracks website")
    bank_names, bank_list_url, bank_logos = scrapping_main_page_bank_track()

    # Initiate geolocator before loop
    geolocator = Nominatim(user_agent="bank_scraper",timeout=10)
    # Initiate list of lines
    rows = []
    for bank_name, bank_url, logo in zip(bank_names, bank_list_url, bank_logos):
        # if bank name not in banks find in BOCC then dont scrap the content
        if bank_name not in banks_in_bocc:
            continue
        LOGGER.debug(f"{bank_name}: found in BOCC banks, scrap details from bank page")
        raw_info = scrapping_description_bank_page(bank_url)
        clean_info = process_raw_info(raw_info)

        address_attempts = [
            f"{clean_info['Headquarters address']}, {clean_info['Headquarters country']}",
            clean_info['Headquarters address'].split(",")[-1].strip(),
            clean_info['Headquarters country']
        ]
        location = None
        for addr in address_attempts:
            LOGGER.debug(f"{bank_name}: trying geopy with '{addr}'")
            location = geolocator.geocode(addr)
            if location:
                break 
        clean_info["Latitude"] = location.latitude if location else None
        clean_info["Longitude"] = location.longitude if location else None
        clean_info["Bank Name"] = bank_name
        clean_info["Source BankTrack"] = bank_url
        clean_info["Bank logo"] = logo
        rows.append(clean_info)

    # Create DataFrame based on list of rows
    df = pd.DataFrame(rows, columns=columns_dataframe + ["Bank logo"])
    
    # Remap some country name
    df["Headquarters country"] = df["Headquarters country"].replace(
        {
            "Taiwan, Republic of China": "Taiwan",
            "Russian Federation": "Russia",
        }
    )

    # Add World Region associated to Headquarters country
    LOGGER.debug("Get world region using Headquarters country column")
    df["World Region"] = df["Headquarters country"].apply(get_world_region)

    # sort df
    LOGGER.debug("Sort dataset by bank name")
    df = df.sort_values(by="Bank Name", ascending=True)
    
    # replace back bank name to be coherent with BOCC
    # invert the dictionary (swap keys and values)
    reverse_match_bank = {v: k for k, v in manual_match_bank.items()}
    df["Bank Name"] = df["Bank Name"].replace(reverse_match_bank)
    
    # rename column to snake_case 
    rename_dict = {
        "Bank Name": "bank_name",
        "Bank Website": "bank_website",
        "Headquarters address": "headquarters_address",
        "Headquarters country": "headquarters_country",
        "CEO Name": "ceo_name",
        "Board description": "board_description",
        "Supervisor Name": "supervisor_name",
        "Supervisor Website": "supervisor_website",
        "Shareholder structure source": "shareholder_structure_source",
        "Source BankTrack": "source_banktrack",
        "Latitude": "latitude",
        "Longitude": "longitude",
        "Bank logo": "bank_logo",
        "World Region": "world_region"
    }
    df = df.rename(columns=rename_dict)
    # Manualy add informations on Capital One Financial : 
    new_row = {
    "bank_name": "Capital One Financial",
    "bank_website": "None",
    "headquarters_address": "McLean, Virginie",
    "headquarters_country": "United States",
    "ceo_name": "None",
    "board_description": "None",
    "supervisor_name": "None",
    "supervisor_website": "None",
    "shareholder_structure_source": "None",
    "source_banktrack": "None",
    "latitude": "38.938305",
    "longitude": "-77.183266",
    "bank_logo": "None",
    "world_region": "None"
    }
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    # Return dataframe with all info on bank companies
    return df

In [15]:
#df_bank_data = pd.DataFrame()
df_bank_data = create_banks_table(df_bocc)

# Processing on BOCC file

In [ ]:
renamed_bocc_columns = {
    'Bank': 'bank',
    'Company': 'company',
    2021: '2021',
    2022: '2022',
    2023: '2023',
    2024: '2024',
    'GOGEL ID': 'company_gogel_id'
}
df_bocc.rename(columns=renamed_bocc_columns, inplace=True)

# Uniform company name to match Rystad name
df_filtered = df_companies_involvement[['company','gogel_id']].copy()
df_filtered['company'] = df_filtered['company'].str.strip()
df_filtered['gogel_id'] = df_filtered['gogel_id'].astype(str).str.strip()

df_filtered.drop_duplicates(subset= ['company','gogel_id'], inplace = True)

df_filtered = df_filtered.rename(columns={'gogel_id': 'company_gogel_id'})

df_bocc['company_gogel_id'] = df_bocc['company_gogel_id'].astype(str)
mapping = dict(zip(df_filtered['company_gogel_id'], df_filtered['company']))
# Replace company names in df_bocc where there is a match
df_bocc['company'] = df_bocc['company_gogel_id'].map(mapping).combine_first(df_bocc['company'])

# Reorder df_gasoil_emissions columns

In [17]:
final_order_df_gasoil_emissions = [
    'project_name',
    'country',
    'latitude',
    'longitude',
    'world_region',
    'start_year',
    'total_potential_emissions_v1',
    'total_potential_emissions',
    'producing_potential_emissions',
    'short_term_expansion_potential_emissions',
    'long_term_expansion_potential_emissions',
    'project_evolution',
    'project_type',
    'new_extraction_after_2021',
    'project_status_in_v1',
    'project_status',
    'list_of_company_involved',
    'fuel_type',
    '2020_estimated_emissions',
    '2021_estimated_emissions',
    '2022_estimated_emissions',
    '2023_estimated_emissions',
    '2024_estimated_emissions',
    '2025_estimated_emissions',
    '2020_2025_past_emissions',
    'percentage_evolution',
    'percentage_emissions_new_extraction',
    'project_name_raw'
    ]

df_gasoil_emissions = df_gasoil_emissions[final_order_df_gasoil_emissions]

# Create Excel output file


In [18]:
CONFIDENTIAL = False

# Mapping DataFrames to sheets names
dataframes_to_export = {
    "df_gasoil_emissions": "gasoil_project_data",
    "df_companies_data": "company_data",
    "df_bank_data" : "bank_data",
    "df_bocc": "connection_bank_company",
    "df_companies_involvement": "connection_project_company",
    "df_lng": "lng_data",
    "df_countries": "country_data",
}

# Columns to drop for the PUBLIC version
df_gasoil_emissions_columns_to_filter = [
    "2020_estimated_emissions",
    "2021_estimated_emissions",
    "2022_estimated_emissions",
    "2023_estimated_emissions",
    "2024_estimated_emissions",
    "2025_estimated_emissions",
    "percentage_evolution",
    "percentage_emissions_new_extraction",
    "project_name_raw",
]

df_companies_data_columns_to_filter = [
    "gogel_id",
    "company_isin",
    "company_lei"
]

df_companies_involvement_columns_to_filter = [
    "gogel_id",
    "company_involvement_percent",
]

df_bocc_columns_to_filter = [
    "company_gogel_id"
]


def export_dataframes(file_name: str, public: bool = False):
    """
    Export all dataframes to an Excel file.
    If public=True, sensitive columns from df_gasoil_emissions are dropped.
    Also creates a 'metadatas' sheet with all sheet_name/column_name pairs.
    """
    metadata_records = [] # When commented, do not created empty metadatas sheet 
    

    with pd.ExcelWriter(file_name, engine="xlsxwriter") as writer:
        for df_var_name, sheet_name in dataframes_to_export.items():
            df = globals().get(df_var_name)
            if df is not None:
                df_to_export = df.copy()

                # Drop sensitive columns only in PUBLIC version
                if public and df_var_name == "df_gasoil_emissions":
                    df_to_export = df_to_export.drop(
                        columns=df_gasoil_emissions_columns_to_filter,
                        errors="ignore"
                    )
                    
                if public and df_var_name == "df_companies_data":
                    df_to_export = df_to_export.drop(
                        columns=df_companies_data_columns_to_filter,
                        errors="ignore"
                    )
                    
                if public and df_var_name == "df_companies_involvement":
                    df_to_export = df_to_export.drop(
                        columns=df_companies_involvement_columns_to_filter,
                        errors="ignore"
                    )
                    
                if public and df_var_name == "df_bocc":
                    df_to_export = df_to_export.drop(
                        columns=df_bocc_columns_to_filter,
                        errors="ignore"
                    )

                # Save DataFrame to Excel
                df_to_export.to_excel(writer, sheet_name=sheet_name, index=False)

                # Add metadata (sheet_name, column_name)
                for col in df_to_export.columns:
                    metadata_records.append({"sheet_name": sheet_name, "column_name": col})
            else:
                print(f"Warning: {df_var_name} not found in global variables.")

        # Create metadata DataFrame and export it as last sheet
        if metadata_records:
            df_metadata_completed = pd.read_csv("../data_sources/metadatas.csv",sep = ";")
            df_metadata = pd.DataFrame(metadata_records)
            df_metadata = df_metadata.merge(
                df_metadata_completed[["sheet_name", "column_name", "definition", "sources"]],
                on=["sheet_name", "column_name"],
                how="left"
            )
            df_metadata.to_excel(writer, sheet_name="metadatas", index=False)

    print(f"✅ DataFrames saved to {file_name}")


# Generate both files
export_dataframes("output_data_CONFIDENTIAL.xlsx", public=False)
export_dataframes("output_data_PUBLIC.xlsx", public=True)

✅ DataFrames saved to output_data_CONFIDENTIAL.xlsx
✅ DataFrames saved to output_data_PUBLIC.xlsx


# TESTS


### Tests on relationship between dataframe

In [19]:
# Test BOCC / Bank data
set_bocc_bank = set(df_bocc['bank'].unique())
set_bank_data = set(df_bank_data['bank_name'].unique())
only_in_bocc_bank = set_bocc_bank - set_bank_data
only_in_bank_data = set_bank_data - set_bocc_bank

# Test Connexion project company / Gasoil project 
set_connexion = set(
    df_companies_involvement
    .loc[df_companies_involvement["project_type"] != "New LNG terminal", "project_name"]
    .unique()
)
set_project = set(df_gasoil_emissions['project_name_raw'].unique())
only_in_project = set_project - set_connexion
only_in_connexion = set_connexion - set_project

test_relationship = 'pass'
dict_diff_set = {
	"only_in_bocc": only_in_bocc_bank,
	"only_in_bank_data": only_in_bank_data,
	# "only_in_project":only_in_project, # OK for US to have project without data for connexion
	"only_in_connexion":only_in_connexion,
}

for name, elt in dict_diff_set.items() : 
	if elt != set():
		print(f"Warning on {name}, not empty : {elt}")
		test_relationship = 'failed'
print(f"Test relationship : {test_relationship}")

Test relationship : pass


### Non null test

In [20]:
def check_nulls(df, columns, dataset_name, condition=None):
    """
    Checks for null values in specific columns of a DataFrame.
    
    Parameters:
        df (pd.DataFrame): The DataFrame to check
        columns (list): Columns to validate
        dataset_name (str): Name of the dataset for error reporting
        condition (pd.Series, optional): Boolean mask to filter rows before checking
    """
    if condition is not None:
        df = df.loc[condition]

    if df[columns].isnull().any().any():
        print(f"ERROR: Null values detected for {dataset_name}")
    else:
        print(f"No null values detected for {dataset_name}")


# Configuration: dataset, columns, and optional filter condition
datasets_to_check = [
    {
        "df": df_gasoil_emissions,
        "columns": [
            'project_name', 'country', 'latitude', 'longitude', 'world_region',
            'total_potential_emissions', 'producing_potential_emissions',
            'short_term_expansion_potential_emissions',
            'long_term_expansion_potential_emissions', 'project_evolution',
            'project_type', 'new_extraction_after_2021', 'project_status_in_v1',
            'project_status', 'fuel_type', 'percentage_evolution',
            'project_name_raw'
        ],
        "name": "gasoil projects",
        "condition": lambda df: df['project_status'] != 'project not found'
    },
    {
        "df": df_companies_data,
        "columns": [
            'company', 'headquarter_country', 'latitude', 'longitude',
            'world_region', 'gogel_id'
        ],
        "name": "companies data"
    },
    {
        "df": df_bocc,
        "columns": ['bank', 'company', 'company_gogel_id'],
        "name": "bocc data"
    },
    {
        "df": df_lng,
        "columns": [
            'project_name', 'export_capacity_in_mtpa', 'project_status',
            'country', 'companies_involved', 'latitude', 'longitude'
        ],
        "name": "lng data"
    },
    {
        "df": df_countries,
        "columns": [
            'country', 'Emissions_per_capita_tons_CO2',
            'Year_Emissions_per_capita_tons_CO2',
            'Emissions_thousand_tons_CO2',
            'Year_Emissions_thousand_tons_CO2',
            'GDP_millions_US_dollars',
            'Year_GDP_millions_US_dollars',
            'GDP_per_capita_US_dollars',
            'Year_GDP_per_capita_US_dollars',
            'Population_in_millions',
            'Year_Population_in_millions',
            'Surface_thousand_km2',
            'Year_Surface_thousand_km2'
        ],
        "name": "countries data"
    },
    {
        "df": df_bank_data,
        "columns": [
            'bank_name', 'bank_website', 'headquarters_address',
            'headquarters_country', 'ceo_name', 'board_description',
            'supervisor_name', 'supervisor_website',
            'shareholder_structure_source', 'source_banktrack',
            'latitude', 'longitude', 'bank_logo', 'world_region'
        ],
        "name": "banks data"
    },
    {
        "df": df_companies_involvement,
        "columns": ['project_name', 'company', 'project_type'],
        "name": "companies_involvement data"
    }
]

# Run checks
for dataset in datasets_to_check:
    condition = dataset.get("condition")
    check_nulls(
        df=dataset["df"],
        columns=dataset["columns"],
        dataset_name=dataset["name"],
        condition=condition(dataset["df"]) if condition else None
    )



No null values detected for gasoil projects
No null values detected for companies data
No null values detected for bocc data
No null values detected for lng data
No null values detected for countries data
No null values detected for banks data
No null values detected for companies_involvement data


### Duplicates values

In [21]:
duplicates = df_bocc[df_bocc.duplicated(subset=['bank','company'], keep=False)]
duplicates.to_csv('temp.csv')

In [22]:
def check_duplicates(df, subset, label):
    duplicates = df[df.duplicated(subset=subset, keep=False)]
    
    if not duplicates.empty:
        print(f"❌ Error: Found duplicated rows in {label} based on {subset}:")
        print(duplicates)
    else:
        print(f"✅ Success: No duplicates found in {label} based on {subset}.")


check_duplicates(df_gasoil_emissions[df_gasoil_emissions["project_evolution"] != "project_not_found"], ['project_name_raw'], "df_gasoil_emissions")
check_duplicates(df_companies_involvement, ['project_name', 'company'], "df_companies_involvement")
check_duplicates(df_companies_data, ['company','headquarter_country'], "df_companies_data")
check_duplicates(df_bank_data, ['bank_name'], "df_bank_data")
check_duplicates(df_bocc, ['bank','company'], "df_bocc")
check_duplicates(df_lng, ['project_name'], "df_lng")
check_duplicates(df_countries, ['country'], "df_countries")





✅ Success: No duplicates found in df_gasoil_emissions based on ['project_name_raw'].
✅ Success: No duplicates found in df_companies_involvement based on ['project_name', 'company'].
✅ Success: No duplicates found in df_companies_data based on ['company', 'headquarter_country'].
✅ Success: No duplicates found in df_bank_data based on ['bank_name'].
❌ Error: Found duplicated rows in df_bocc based on ['bank', 'company']:
                                         bank  \
130                                      HSBC   
455                            JPMorgan Chase   
456                                 Citigroup   
457                           Bank of America   
458                          Mizuho Financial   
...                                       ...   
11130  Banco Bilbao Vizcaya Argentaria (BBVA)   
11492                   Capital One Financial   
11506                   Capital One Financial   
11534                   Capital One Financial   
11549                   Capital One Fin

### Others test 


In [23]:
# connection_project_company groupby project = 100%
tolerance = 0.01
df_filtered = df_companies_involvement[
    df_companies_involvement["project_type"] != "New LNG terminal"
]
check = (
    df_filtered
    .groupby("project_name")["company_involvement_percent"]
    .sum()
)
invalid_projects = check[(check < 100 - tolerance) | (check > 100 + tolerance)]
if invalid_projects.empty:
    print("✅ All projects sum to ~100 (within tolerance)")
else:
    print("❌ These projects do not sum to 100 ±0.01:")
    print("Explanation for those project : They have NULL emissions for each company marked as involved. Hence division by 0 lead to unknown")
    for project, total in invalid_projects.items():
        print(f"  - {project}: {total}")



❌ These projects do not sum to 100 ±0.01:
Explanation for those project : They have NULL emissions for each company marked as involved. Hence division by 0 lead to unknown
  - A Cluster, MY: 0.0
  - Alakiri (redevelop), NG: 0.0
  - Anhanga, BR: 0.0
  - Bosi, NG: 0.0
  - Cendana, ID: 0.0
  - King Street, US: 0.0
  - Lengo Gas Field, ID: 0.0
  - Leopard, GA: 0.0
  - NC008A (El Hamra I), LY: 0.0
  - Naajal, MX: 0.0
  - Nene Marine, CG: 0.0
  - Okoloma, NG: 0.0
  - Sockeye-2, US: 0.0


In [24]:

# Check if latitude or longitude equals 0
for name, df in {
    "df_gasoil_emissions": df_gasoil_emissions,
    "df_companies_data": df_companies_data,
    "df_lng": df_lng,
    "df_bank_data": df_bank_data,
}.items():
    print(f"\n{name}")
    print("Latitude == 0:", (df['latitude'] == 0).sum())
    print("Longitude == 0:", (df['longitude'] == 0).sum())
    print("Any row with lat=0 and long=0:", ((df['latitude'] == 0) & (df['longitude'] == 0)).sum())



df_gasoil_emissions
Latitude == 0: 0
Longitude == 0: 1
Any row with lat=0 and long=0: 0

df_companies_data
Latitude == 0: 0
Longitude == 0: 0
Any row with lat=0 and long=0: 0

df_lng
Latitude == 0: 0
Longitude == 0: 0
Any row with lat=0 and long=0: 0

df_bank_data
Latitude == 0: 0
Longitude == 0: 0
Any row with lat=0 and long=0: 0


In [25]:
# Expected values
expected_v2 = 31
expected_v1 = 195

# Actual counts
count_v2 = df_gasoil_emissions[df_gasoil_emissions["project_evolution"] == "new_identified_carbon_bomb"].shape[0]
count_v1 = df_gasoil_emissions[
    (df_gasoil_emissions["project_type"] == "Carbon bombs") &
    (df_gasoil_emissions["project_evolution"] != "new_identified_carbon_bomb")
].shape[0]

# Checks with print
if count_v2 != expected_v2:
    print(f"❌ Error: Count v2 mismatch — expected {expected_v2}, got {count_v2}")
else:
    print(f"✅ Count v2 OK: {count_v2}")

if count_v1 != expected_v1:
    print(f"❌ Error: Count v1 mismatch — expected {expected_v1}, got {count_v1}")
else:
    print(f"✅ Count v1 OK: {count_v1}")


✅ Count v2 OK: 31
✅ Count v1 OK: 195


In [26]:
# Test that No empty list_of_company_involved in df_gasoil_emissions when project_type == "Carbon Bombs" and project_evolution != "project_not_found"

# Select rows with conditions
df_test = df_gasoil_emissions[
    (df_gasoil_emissions['project_type'] == 'Carbon bombs') &
    (df_gasoil_emissions['project_evolution'] != 'project_not_found')
]

# Check for nulls
if df_test['list_of_company_involved'].isnull().any():
    print("⚠️ Empty 'list_of_company_involved' detected!")
    print(df_test.loc[df_test['list_of_company_involved'].isnull(),'project_name_raw'])
else:
    print("✅ No empty 'list_of_company_involved' found.")



✅ No empty 'list_of_company_involved' found.
